# Lesson 2A: Dynamic Programming - Theory

## Introduction

In Lesson 1, we learned the **mathematical foundations** of MDPs (Bellman equations, value functions, optimal policies). In Lesson 1B, we **implemented** policy iteration and value iteration on FrozenLake.

Now we go **deep** into Dynamic Programming—the theory that explains *why* these algorithms work and *when* they're optimal.

### What is Dynamic Programming?

**Dynamic Programming (DP)** is a class of algorithms that:
1. Exploit **Markov structure** (memoryless transitions)
2. Break problems into **overlapping subproblems** (Bellman recursion)
3. Use **memoization** (store computed values, reuse them)

Classic DP example: Computing Fibonacci numbers
- Naive recursive: F(n) = F(n-1) + F(n-2) → exponential time O(2^n)
- DP: Store F(i) for i=1..n, reuse them → linear time O(n)

In RL, the **Bellman equations** provide the recursive structure, and DP algorithms solve them efficiently.

### Learning Objectives

1. Understand **prediction vs. control** problems
2. Master the **policy improvement theorem**
3. Derive **policy iteration** rigorously
4. Derive **value iteration** rigorously
5. Analyze **convergence** guarantees
6. Understand **computational complexity**
7. Know when DP is applicable (and when it's not)

## Table of Contents

1. [Setup](#setup)
2. [Prediction vs. Control](#prediction)
3. [Policy Improvement Theorem](#pie)
4. [Policy Iteration (Derivation)](#policy-iter)
5. [Value Iteration (Derivation)](#value-iter)
6. [Convergence Theory](#convergence)
7. [Computational Complexity](#complexity)
8. [Generalized Policy Iteration](#gpi)
9. [Implementation Deep Dive](#impl)
10. [Advanced Topics](#advanced)
11. [Summary](#summary)

<a name="setup"></a>
## Setup

In [ ]:
import subprocess
import sys

packages = ['numpy', 'matplotlib', 'seaborn', 'pandas', 'scipy']
for pkg in packages:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Rectangle
import seaborn as sns
import pandas as pd
from scipy.linalg import solve
from typing import Dict, List, Tuple, Callable

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ All libraries loaded!')

<a name="prediction"></a>
## Prediction vs. Control

### Two Fundamental Problems

DP methods solve two related problems:

#### 1. **Prediction Problem**

Given:
- MDP model (S, A, P, R, γ)
- Fixed policy π

Compute:
- Value function V^π(s) for all states s

**Question**: How good is this policy? What's the expected return starting from each state?

**Algorithm**: **Policy Evaluation** (iteratively apply Bellman expectation equation)

#### 2. **Control Problem**

Given:
- MDP model (S, A, P, R, γ)

Compute:
- Optimal value function V*(s) for all states s
- Optimal policy π*(s) for all states s

**Question**: What's the best policy? What should the agent do to maximize return?

**Algorithm**: **Policy Iteration** or **Value Iteration** (iteratively improve policy)

### The Connection

Both use the **Bellman equations** as their core:
- **Expectation equation** (prediction): Given π, compute V^π
- **Optimality equation** (control): Find π* and V*

**Key insight**: Prediction is a subproblem of control. Control algorithms repeatedly solve prediction problems and improve the policy.

In [ ]:
# Visualize prediction vs. control
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Prediction
ax = axes[0]
ax.text(0.5, 0.95, 'PREDICTION PROBLEM', ha='center', fontsize=13, fontweight='bold',
        transform=ax.transAxes)

ax.text(0.5, 0.85, 'Given: Policy π', ha='center', fontsize=11, transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

ax.text(0.5, 0.70, 'Apply Bellman\nExpectation Eq.', ha='center', fontsize=10, transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.text(0.15, 0.70, 'V^π(s) =', ha='center', fontsize=10, family='monospace', transform=ax.transAxes)
ax.text(0.5, 0.58, r'$\sum_a \pi(a|s) \sum_{s\'} P(s\'|s,a)[r + \gamma V^\pi(s\'')]$',
       ha='center', fontsize=11, transform=ax.transAxes)

ax.text(0.5, 0.40, 'Compute\nValue Function', ha='center', fontsize=10, transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

ax.text(0.5, 0.20, 'Output: V^π(s) for all states', ha='center', fontsize=10, transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))

ax.text(0.5, 0.05, 'Used by: Policy Evaluation', ha='center', fontsize=9, style='italic', transform=ax.transAxes)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# Control
ax = axes[1]
ax.text(0.5, 0.95, 'CONTROL PROBLEM', ha='center', fontsize=13, fontweight='bold',
        transform=ax.transAxes)

ax.text(0.5, 0.85, 'Given: MDP only', ha='center', fontsize=11, transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

ax.text(0.5, 0.72, '① Evaluate current policy π', ha='center', fontsize=10, transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='#ffffcc', alpha=0.8))

ax.text(0.5, 0.60, '② Greedy improvement:\nπ\'(s) = argmax_a Q(s,a)', ha='center', fontsize=10,
       transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='#ffffcc', alpha=0.8))

ax.text(0.5, 0.42, 'Repeat until π = π\'\n(policy convergence)', ha='center', fontsize=10, transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.text(0.5, 0.25, 'Output: π*(s) and V*(s)', ha='center', fontsize=10, transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))

ax.text(0.5, 0.10, 'Used by: Policy Iteration,', ha='center', fontsize=9, style='italic', transform=ax.transAxes)
ax.text(0.5, 0.05, 'Value Iteration', ha='center', fontsize=9, style='italic', transform=ax.transAxes)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.show()

<a name="pie"></a>
## Policy Improvement Theorem

### The Core Theorem

**Theorem (Policy Improvement Theorem)**:

Given two policies π and π', if:
$$Q^\pi(s, \pi'(s)) \geq V^\pi(s) \quad \forall s \in S$$

Then:
$$V^{\pi'}(s) \geq V^\pi(s) \quad \forall s \in S$$

**In words**: If the new policy is at least as good as the old policy in *one step* (looking ahead via Q-values), then it's at least as good overall.

### Why This Matters

This justifies the greedy policy improvement step:
$$\pi'(s) = \arg\max_a Q^\pi(s, a)$$

This guaranteed improvement is the foundation of all policy iteration algorithms.

### Proof Sketch

Starting from state s with π':

\begin{align}
V^\pi(s) &\leq Q^\pi(s, \pi'(s)) && \text{(by assumption)} \\
&= \mathbb{E}_{\pi'}[R_{t+1} + \gamma V^\pi(S_{t+1}) | S_t = s] && \text{(def of Q)} \\
&\leq \mathbb{E}_{\pi'}[R_{t+1} + \gamma Q^\pi(S_{t+1}, \pi'(S_{t+1})) | S_t = s] && \text{(apply assumption)} \\
&\leq V^{\pi'}(s) && \text{(by induction)} 
\end{align}

The key is that assumption holds at each state, so by induction the entire trajectory improves.

In [ ]:
# Visualize policy improvement
fig, ax = plt.subplots(figsize=(12, 6))

ax.text(0.5, 0.95, 'Policy Improvement Theorem', ha='center', fontsize=14, fontweight='bold',
        transform=ax.transAxes)

# Current policy
ax.text(0.15, 0.80, 'Current Policy π', ha='center', fontsize=11, fontweight='bold',
        transform=ax.transAxes)
ax.text(0.15, 0.70, 'V^π(s) = 0.5', ha='center', fontsize=10, family='monospace',
        transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='#ffffcc', alpha=0.9))

# Q-values
ax.text(0.50, 0.80, 'Q^π(s, a) values', ha='center', fontsize=11, fontweight='bold',
        transform=ax.transAxes)
ax.text(0.50, 0.70, 'Left:  Q^π(s, ←) = 0.3', ha='center', fontsize=10, family='monospace',
        transform=ax.transAxes)
ax.text(0.50, 0.63, 'Down:  Q^π(s, ↓) = 0.8', ha='center', fontsize=10, family='monospace',
        transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='#ccffcc', alpha=0.9))
ax.text(0.50, 0.56, 'Right: Q^π(s, →) = 0.4', ha='center', fontsize=10, family='monospace',
        transform=ax.transAxes)
ax.text(0.50, 0.49, 'Up:    Q^π(s, ↑) = 0.2', ha='center', fontsize=10, family='monospace',
        transform=ax.transAxes)

# Arrow
ax.annotate('', xy=(0.27, 0.65), xytext=(0.38, 0.65),
           arrowprops=dict(arrowstyle='->', lw=3, color='blue'), xycoords=ax.transAxes)
ax.text(0.325, 0.68, 'Check', ha='center', fontsize=10, color='blue', fontweight='bold',
        transform=ax.transAxes)

# New policy
ax.text(0.85, 0.80, 'New Policy π\'', ha='center', fontsize=11, fontweight='bold',
        transform=ax.transAxes)
ax.text(0.85, 0.70, 'π\'(s) = argmax\n        = ↓', ha='center', fontsize=10, family='monospace',
        transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='#ccffcc', alpha=0.9))
ax.text(0.85, 0.57, 'V^π\'(s) ≥ 0.5', ha='center', fontsize=10, family='monospace',
        transform=ax.transAxes)
ax.text(0.85, 0.48, '✓ Guaranteed\nImprovement', ha='center', fontsize=10,
        transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.9))

# Box highlighting the condition
rect = Rectangle((0.32, 0.38), 0.36, 0.18, transform=ax.transAxes, 
                 linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
ax.add_patch(rect)

ax.text(0.5, 0.32, 'Condition: All Q^π(s, a) ≥ V^π(s)?\nYes → π\' is better or equal', ha='center', fontsize=10,
        transform=ax.transAxes, color='red', fontweight='bold')

ax.text(0.5, 0.15, 'Example: max(0.3, 0.8, 0.4, 0.2) = 0.8 ≥ 0.5 ✓', ha='center', fontsize=11,
        transform=ax.transAxes, family='monospace',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

ax.text(0.5, 0.03, 'This principle drives policy improvement in all DP algorithms', ha='center', fontsize=10,
        transform=ax.transAxes, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.show()

<a name="policy-iter"></a>
## Policy Iteration - Derivation

### Algorithm Overview

Policy Iteration alternates two steps:

```
Initialize π arbitrarily, V arbitrarily

loop:
  1. Policy Evaluation: Compute V^π using Bellman expectation equation
  2. Policy Improvement: π := greedy(π, V)
  until π doesn't change

return π*, V*
```

### Convergence Guarantees

**Theorem**: Policy Iteration converges to π* in **finite iterations**.

**Proof intuition**:
1. After each policy evaluation, we have the true value V^π for current policy
2. Greedy improvement guarantees V^{π_new}(s) ≥ V^π(s) for all s
3. Values form a discrete lattice (finite state space)
4. Monotonic improvement with finite upper bound → finite convergence
5. When policy stops changing, we have π* (proven by optimality condition)

### Computational Complexity

- **Iterations**: O(n) policy iterations in worst case (typically 2-10)
- **Per iteration**:
  - Policy Evaluation: O(n²) or O(n³) depending on implementation
  - Policy Improvement: O(n·m) where m = # actions
- **Total**: O(n³) per policy iteration, converges in O(n) iterations

### Optimality Condition

When Policy Improvement yields π' = π (no change), we have:
$$\pi(s) = \arg\max_a Q^\pi(s, a)$$

This means:
$$V^\pi(s) = \max_a Q^\pi(s, a) = \max_a \sum_{s',r} P(s',r|s,a)[r + \gamma V^\pi(s')]$$

Which is exactly the **Bellman optimality equation**! So π = π*.

In [ ]:
# Flowchart of Policy Iteration
fig, ax = plt.subplots(figsize=(10, 10))

def draw_box(ax, x, y, width, height, text, color='lightblue'):
    rect = Rectangle((x-width/2, y-height/2), width, height, 
                     facecolor=color, edgecolor='black', linewidth=2, transform=ax.transAxes)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=10, transform=ax.transAxes,
           fontweight='bold')

def draw_diamond(ax, x, y, width, height, text, color='lightyellow'):
    from matplotlib.patches import FancyBboxPatch
    rect = FancyBboxPatch((x-width/2, y-height/2), width, height,
                         boxstyle='round,pad=0.02', facecolor=color, 
                         edgecolor='black', linewidth=2, transform=ax.transAxes)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=9, transform=ax.transAxes,
           fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
               arrowprops=dict(arrowstyle='->', lw=2, color='black'), 
               xycoords=ax.transAxes)
    if label:
        mid_x, mid_y = (x1+x2)/2, (y1+y2)/2
        ax.text(mid_x+0.05, mid_y, label, fontsize=9, transform=ax.transAxes,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

# Draw boxes and arrows
draw_box(ax, 0.5, 0.95, 0.25, 0.05, 'START', 'lightgreen')
draw_arrow(ax, 0.5, 0.925, 0.5, 0.85)

draw_box(ax, 0.5, 0.80, 0.4, 0.08, 'Initialize: π, V', 'lightblue')
draw_arrow(ax, 0.5, 0.76, 0.5, 0.70)

draw_box(ax, 0.5, 0.65, 0.35, 0.08, 'Policy Evaluation', 'lightyellow')
ax.text(0.5, 0.57, 'Compute V^π(s) using\nBellman expectation eq.', ha='center', fontsize=8,
       transform=ax.transAxes, style='italic')
draw_arrow(ax, 0.5, 0.53, 0.5, 0.47)

draw_box(ax, 0.5, 0.42, 0.35, 0.08, 'Policy Improvement', 'lightyellow')
ax.text(0.5, 0.34, 'π\' ← greedy(V^π)', ha='center', fontsize=8,
       transform=ax.transAxes, style='italic')
draw_arrow(ax, 0.5, 0.30, 0.5, 0.24)

draw_diamond(ax, 0.5, 0.17, 0.2, 0.08, 'π\' = π?', 'lightyellow')

# Yes path (loop back)
draw_arrow(ax, 0.4, 0.17, 0.15, 0.17, 'No')
draw_arrow(ax, 0.15, 0.17, 0.15, 0.65)
draw_arrow(ax, 0.15, 0.65, 0.325, 0.65)
ax.text(0.12, 0.40, 'Update π', fontsize=8, transform=ax.transAxes, style='italic')

# No path (exit)
draw_arrow(ax, 0.5, 0.125, 0.5, 0.08, 'Yes')
draw_box(ax, 0.5, 0.03, 0.25, 0.05, 'Return π*', 'lightgreen')

ax.text(0.5, 0.99, 'Policy Iteration Algorithm', ha='center', fontsize=13, fontweight='bold',
       transform=ax.transAxes)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.show()

print('Policy Iteration: Two-phase approach')
print('Phase 1: Evaluate current policy (solve Bellman eq.)')
print('Phase 2: Improve policy greedily')
print('Loop: Repeat until policy stabilizes')

<a name="value-iter"></a>
## Value Iteration - Derivation

### The Key Insight

Instead of explicitly evaluating policies, Value Iteration directly solves the **Bellman optimality equation**:

$$V(s) = \max_a \sum_{s',r} P(s',r|s,a)[r + \gamma V(s')]$$

The algorithm:

```
Initialize V arbitrarily

loop:
  for each state s:
    V(s) ← max_a sum_{s',r} P(s',r|s,a)[r + γV(s')]
  until V converges

Extract policy: π(s) = argmax_a ...
return π*, V*
```

### Why This Works

**Theorem (Contraction Mapping)**: The Bellman optimality operator T is a **contraction**:

$$\|TV_1 - TV_2\|_\infty \leq \gamma \|V_1 - V_2\|_\infty \quad \forall V_1, V_2$$

where $T V(s) = \max_a \sum_{s',r} P(s',r|s,a)[r + \gamma V(s')]$

**Consequence**: By Banach Fixed Point Theorem, repeated application converges to unique fixed point V*.

### Convergence Rate

**Geometric convergence**: Error decreases by factor γ each iteration:

$$\|V_k - V^*\|_\infty \leq \gamma^k \|V_0 - V^*\|_\infty$$

To reduce error by factor 10^{-d}, need:
$$\gamma^k \leq 10^{-d} \Rightarrow k \geq \frac{d \log 10}{-\log \gamma}$$

Example: γ = 0.9, d = 6 (6 decimal places):
$$k \geq \frac{6 \times 2.3}{2.1} \approx 6.6 \text{ iterations}$$

### Complexity

- **Iterations**: O(log(1/epsilon)) to reach epsilon-optimal solution
- **Per iteration**: O(n²m) for n states, m actions
- **Total**: Better than Policy Iteration for large problems

In [ ]:
# Visualize Bellman optimality operator as contraction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Contraction visualization
ax = axes[0]
ax.text(0.5, 0.95, 'Bellman Operator: Contraction Mapping', ha='center', fontsize=12, fontweight='bold',
        transform=ax.transAxes)

# Example: Simple 1D visualization
x = np.linspace(0, 10, 100)
gamma = 0.9
V_optimal = 5  # Suppose optimal V* = 5

# T operator: TV = gamma*(x-5) + 5 (simplified)
TV = gamma * (x - V_optimal) + V_optimal

ax.plot(x, x, 'k--', linewidth=2, label='Identity (V = V)', alpha=0.5)
ax.plot(x, TV, 'b-', linewidth=2.5, label=f'T(V) = γ(V-V*) + V* (γ={gamma})')
ax.plot([V_optimal], [V_optimal], 'ro', markersize=10, label='Fixed point V*')

# Show iteration
V0 = 8
colors = plt.cm.viridis(np.linspace(0, 1, 5))
for i in range(4):
    V_new = gamma * (V0 - V_optimal) + V_optimal
    ax.plot([V0, V0], [V0, V_new], color=colors[i], linewidth=1.5, alpha=0.7)
    ax.plot([V0, V_new], [V_new, V_new], color=colors[i], linewidth=1.5, alpha=0.7)
    ax.plot([V0], [V_new], 'o', color=colors[i], markersize=6)
    V0 = V_new

ax.set_xlabel('V(s)', fontsize=11)
ax.set_ylabel('T(V(s))', fontsize=11)
ax.set_xlim(4, 10)
ax.set_ylim(4, 10)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.text(0.5, -0.15, 'Repeated application: V_k → V*', ha='center', fontsize=10, transform=ax.transAxes)

# Right: Convergence rate
ax = axes[1]
ax.text(0.5, 0.95, 'Geometric Convergence Rate', ha='center', fontsize=12, fontweight='bold',
        transform=ax.transAxes)

gammas_plot = [0.5, 0.7, 0.9, 0.95, 0.99]
iterations = np.arange(0, 51)

for gamma_val in gammas_plot:
    error = gamma_val ** iterations
    ax.semilogy(iterations, error, 'o-', linewidth=2, markersize=4, label=f'γ={gamma_val}')

ax.axhline(y=1e-6, color='red', linestyle='--', linewidth=2, label='ε=10⁻⁶')
ax.set_xlabel('Iteration k', fontsize=11)
ax.set_ylabel('Error ||V_k - V*|| (log scale)', fontsize=11)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3, which='both')
ax.set_ylim(1e-8, 1e1)
ax.text(0.5, -0.15, 'Higher γ → slower convergence (more discounting = more iterations)', 
       ha='center', fontsize=9, transform=ax.transAxes, style='italic')

plt.tight_layout()
plt.show()

<a name="convergence"></a>
## Convergence Theory

### Mathematical Framework

Both Policy Iteration and Value Iteration are instances of **Generalized Policy Iteration (GPI)**.

#### Theorem (Convergence of GPI)

If both evaluation and improvement processes terminate, then GPI converges to π*.

**Conditions**:
1. Policy Evaluation: Compute or approximate V^π using Bellman equation
2. Policy Improvement: Make policy greedy w.r.t. V
3. Both processes iterate (interact)

**Result**: Convergence is guaranteed under mild conditions (finite state/action spaces).

### When Convergence Fails

Assumptions for convergence:
- ✓ Finite state/action spaces
- ✓ MDP is ergodic (all states reachable)
- ✓ Discount factor γ < 1 (makes infinite-horizon problem finite-horizon equivalent)

If γ = 1 (no discounting):
- Infinite horizons may not converge
- Requires additional assumptions (e.g., undiscounted, episodic problems)

### Rate of Convergence

| Algorithm | Iterations | Per Iteration | Total |
|-----------|-----------|---------------|-------|
| Policy Iteration | Finite (typ. 2-10) | O(n³) | Fast in practice |
| Value Iteration | Log(1/ε) | O(n²m) | Often faster overall |

**Empirical observation**: Policy Iteration converges in fewer policy iterations, but each iteration is expensive. Value Iteration has more total iterations but is simpler per iteration.

<a name="complexity"></a>
## Computational Complexity

### Space Complexity

- **Value function**: O(n) where n = # states
- **Policy**: O(n)
- **Transition model**: O(n·m + n²) where m = # actions
- **Total**: O(n² + nm)

### Time Complexity

#### Policy Iteration

- **Policy Evaluation**: Solving linear system (Bellman equation)
  - System: $(I - \gamma P_\pi)V = R_\pi$
  - Dense solver: O(n³)
  - Iterative solver: O(n²) per iteration, converges in O(1/ε) iterations
  
- **Policy Improvement**: O(nm)
  - Compute Q(s,a) for all (s,a)
  - Take max per state

- **Number of policy iterations**: Typically O(1) to O(log n)

- **Total**: O(n³) or O(n²) depending on solver

#### Value Iteration

- **Per iteration**: O(nm) to compute T(V)
- **Number of iterations**: O(log(1/ε)) to reach ε-optimal
- **Total**: O(nm · log(1/ε))

### Comparison

For typical problems:
- Policy Iteration: Good for small-medium problems
- Value Iteration: Better for large action spaces
- Both scale poorly to very large state spaces (10⁷+)
  - Solution: Function approximation (Lesson 6)
  - Solution: Sampling-based methods (Lesson 3-5)

In [ ]:
# Complexity comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Problem sizes
n_states = np.array([10, 50, 100, 500, 1000, 5000])
n_actions = 4

# Time complexity (in arbitrary units)
pi_time = n_states**3 / 1e6  # O(n^3)
vi_time = n_states * n_actions * np.log(1 / 1e-6) / 1e3  # O(nm * log(1/eps))

ax = axes[0]
ax.loglog(n_states, pi_time, 'o-', linewidth=2.5, markersize=8, label='Policy Iteration O(n³)', color='blue')
ax.loglog(n_states, vi_time, 's-', linewidth=2.5, markersize=8, label='Value Iteration O(nm log(1/ε))', color='orange')

# Reference lines
ax.loglog(n_states, n_states**2 / 1e4, '--', alpha=0.3, color='gray', label='O(n²) reference')
ax.loglog(n_states, n_states**3 / 1e6, '--', alpha=0.3, color='gray', label='O(n³) reference')

ax.set_xlabel('Number of States (n)', fontsize=11)
ax.set_ylabel('Time Complexity (log scale)', fontsize=11)
ax.set_title('Asymptotic Complexity Comparison', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

# Practical runtime (hypothetical)
ax = axes[1]
pi_practical = (n_states**3 / 1e9) + 0.1  # seconds (n^3 complexity, normalized)
vi_practical = (n_states * n_actions * np.log(1 / 1e-6) / 1e7) + 0.05  # seconds

ax.plot(n_states, pi_practical, 'o-', linewidth=2.5, markersize=8, label='Policy Iteration', color='blue')
ax.plot(n_states, vi_practical, 's-', linewidth=2.5, markersize=8, label='Value Iteration', color='orange')

# Mark practical limits
ax.axhline(y=1, color='red', linestyle='--', alpha=0.5, linewidth=1.5, label='1 second threshold')
ax.axhline(y=60, color='darkred', linestyle='--', alpha=0.5, linewidth=1.5, label='1 minute threshold')

ax.set_xlabel('Number of States (n)', fontsize=11)
ax.set_ylabel('Estimated Runtime (seconds)', fontsize=11)
ax.set_title('Practical Runtime Estimates', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print('Takeaway: For n > 100, Value Iteration typically faster')
print('For n > 10,000, both methods need function approximation')

<a name="gpi"></a>
## Generalized Policy Iteration

### Unification Framework

Policy Iteration and Value Iteration are both instances of **Generalized Policy Iteration (GPI)**:

```
loop:
  EVALUATION: Compute/update V based on current policy
  IMPROVEMENT: Update policy to be greedy w.r.t. V
  until convergence
```

### Varying Degrees of Evaluation

The evaluation step can be:
- **Exact**: Fully solve $(I - \gamma P_\pi)V = R_\pi$ (Policy Iteration)
- **Approximate**: One sweep of Bellman equation per improvement (Value Iteration)
- **Very approximate**: One update per state (In-place updates)
- **Asynchronous**: Update states in any order (Asynchronous DP)

All variants converge to π* under GPI.

### Advantage of GPI

Flexibility: Choose evaluation/improvement trade-off for your problem.
- Need faster convergence? Do more evaluation.
- Need simpler code? Do less evaluation.

### Asynchronous Dynamic Programming

Update values in any order, possibly without computing all states each iteration:

```
loop:
  Pick some state s
  V(s) ← max_a sum_{s'} P(s'|s,a)[r + γV(s')]
  until convergence
```

Advantages:
- No need to update all states each iteration
- Can prioritize states (e.g., frequently visited in episodes)
- Enables asynchronous and distributed implementations

<a name="impl"></a>
## Implementation Deep Dive

### In-Place Updates

A subtle but important detail: should we use old or new values?

**Synchronous** (Value Iteration as usually described):
```python
V_new = V.copy()  # Use old values
for s in states:
  V_new[s] = max_a(...)  # Depends on old V
V = V_new
```

**In-place** (often faster):
```python
for s in states:
  V[s] = max_a(...)  # Depends on newest available V
```

In-place converges **faster** because we use more recent information immediately.

### Numerical Stability

1. **Precision**: Use float64, not float32 (avoid rounding errors)
2. **Convergence criterion**: Check max-norm: max_s |V_new(s) - V(s)| < epsilon
3. **Avoiding NaN**: Check for invalid transitions (prob < 0, not summing to 1)

### Vectorization with NumPy

For large problems, vectorize instead of loops:

```python
# Naive (slow)
for s in range(n_states):
  for a in range(n_actions):
    Q[s,a] = sum(P[s,a] * (R[s,a] + gamma * V))

# Vectorized (fast)
Q = (R + gamma * V[None, None, :]) * P[..., None]
Q = Q.sum(axis=2)
```

In [ ]:
# Show implementation details
print("IMPLEMENTATION PATTERNS\n" + "="*50)

print("\n1. CONVERGENCE CHECKING")
print("-" * 30)
print("# Check convergence with max-norm")
print("delta = np.max(np.abs(V_new - V))")
print("if delta < epsilon:")
print("    break  # Converged")

print("\n2. TERMINAL STATE HANDLING")
print("-" * 30)
print("# Terminal states: no transitions, V = 0")
print("if is_terminal[s]:")
print("    V[s] = 0")
print("else:")
print("    V[s] = max_a(...)")

print("\n3. POLICY EXTRACTION")
print("-" * 30)
print("# Extract policy AFTER convergence")
print("for s in states:")
print("    q_values = [compute_Q(s, a) for a in actions]")
print("    policy[s] = np.argmax(q_values)")

print("\n4. HANDLING TIES IN ARGMAX")
print("-" * 30)
print("# Multiple actions have same Q-value")
print("q_values = np.array([...])")
print("max_q = np.max(q_values)")
print("tied_actions = np.where(q_values == max_q)[0]")
print("policy[s] = np.random.choice(tied_actions)  # Random tie-break")

print("\n5. AVOIDING DIVISION BY ZERO")
print("-" * 30)
print("# Check transition probabilities")
print("for s, a in mdp.P:")
print("    assert np.abs(np.sum(mdp.P[s,a]) - 1.0) < 1e-6")
print("    assert np.all(mdp.P[s,a] >= 0)")

<a name="advanced"></a>
## Advanced Topics

### Asynchronous DP

Update values in arbitrary order (enables:
- Distributed computing
- Prioritized sweeps (focus on important states)
- Integration with learning (experience replay)

**Convergence**: Guaranteed if each state updated infinitely often.

### Prioritized Sweeping

Instead of uniform updates, prioritize states with large Bellman residual:

$$|V(s) - \max_a Q(s, a)|$$

States with large residuals converge faster.

### Function Approximation

For very large state spaces (e.g., images), represent V as:

$$V_\theta(s) = \text{NeuralNetwork}_\theta(s)$$

Update θ to minimize Bellman residual (instead of V directly).

This is the foundation for **Deep RL** (Lessons 6-9).

### Continuous State Spaces

Classical DP assumes discrete states. For continuous S:
- **Discretization**: Partition state space into grid
- **Function approximation**: Neural networks, RBFs, etc.
- **Sampling**: Monte Carlo methods (Lesson 3)

### Stochastic Approximation

When exact transitions P(s'|s,a) are unknown but we can sample:

$$V(s) \leftarrow V(s) + \alpha [r + \gamma V(s') - V(s)]$$

This is the basis of **Temporal Difference Learning** (Lesson 4).

<a name="summary"></a>
## Summary

### Key Concepts

1. **Prediction vs. Control**: Compute V^π or find π*
2. **Policy Improvement Theorem**: Guarantees greedy improvement works
3. **Policy Iteration**: Exact evaluation + greedy improvement → finite convergence
4. **Value Iteration**: Approximate evaluation → geometric convergence
5. **Generalized Policy Iteration**: Unifying framework for both
6. **Contraction Mapping**: Bellman operator is a contraction → convergence guaranteed
7. **Complexity**: O(n³) vs O(nm log(1/ε)) trade-off

### When DP Works

✓ Known MDP dynamics (P, R)
✓ Discrete finite state/action spaces
✓ Moderate problem size (n < 10⁶)
✗ Very large state spaces → use function approximation
✗ Unknown dynamics → use learning methods (Lesson 3-5)
✗ Continuous states/actions → use function approximation

### RL Algorithm Family Tree

```
RL Algorithms
├── Dynamic Programming (this lesson)
│   ├── Policy Iteration
│   └── Value Iteration
├── Monte Carlo Methods (Lesson 3)
│   └── Learn from full episodes
├── Temporal Difference Learning (Lesson 4)
│   ├── Q-Learning
│   └── SARSA
├── Policy Gradient Methods (Lesson 8)
│   └── Optimize policy directly
└── Actor-Critic (Lesson 9)
    └── Combine value and policy methods
```

All build on DP foundations. Understanding DP deeply unlocks all methods.

### Next: Lesson 2B

**Practical** implementation of DP on larger environments.
- GridWorld and Maze domains
- Efficient implementation tricks
- When to choose Policy vs. Value Iteration
- Debugging convergence issues

In [ ]:
# Final summary table
summary_data = {
    'Algorithm': ['Policy Iteration', 'Value Iteration', 'Async DP', 'Prioritized Sweeping'],
    'Policy Iterations': ['~2-10', '∞', 'Variable', 'Variable'],
    'Evaluation': ['Exact', 'One sweep', 'One update', 'One update (prioritized)'],
    'Per-Iteration Time': ['O(n³)', 'O(nm)', 'O(1)', 'O(1)'],
    'Convergence': ['Finite', 'Geometric', 'Asymptotic', 'Asymptotic'],
    'Best For': ['Small problems', 'Medium problems', 'Large problems', 'Very large, distributed']
}

df = pd.DataFrame(summary_data)
print(df.to_string(index=False))
print("\n" + "="*80)
print("✓ Dynamic Programming is the foundation of all RL algorithms")
print("✓ Choose based on problem size and computational constraints")
print("✓ All converge to optimal V* and π*")